# MHC-I Predictor Concordance: MHCflurry vs HLApollo on HCC1395

**Purpose:** Compare the two presentation predictors on the full HCC1395 pipeline output to understand where they agree and disagree.

**Data requirements:**
- HCC1395 annotated VCF: `data/HCC1395_inputs/annotated.expression.vcf.gz`
- HLA types: `data/HCC1395_inputs/optitype_normal_result.tsv` (OptiType format)
- Reference proteome: `data/HCC1395_inputs/Homo_sapiens.GRCh38.pep.all.fa.gz`
- HLApollo binary: `tools/HLApollo/HLA-Apollo` (or Docker image)

Run cells top-to-bottom. Both predictors are run on the same peptide candidates derived from HCC1395 somatic variants.

## 1. Setup and data loading

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from neoantigen_pipeline.candidates.peptide_generator import PeptideGenerator
from neoantigen_pipeline.config import MHCIPredictionConfig, PeptideGenerationConfig
from neoantigen_pipeline.io.hla_reader import HLAReader
from neoantigen_pipeline.io.proteome import ProteomeDB
from neoantigen_pipeline.io.vcf_reader import VCFReader
from neoantigen_pipeline.prediction.hlapollo import HLApolloPredictor
from neoantigen_pipeline.prediction.mhcflurry import MHCflurryPredictor

sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

# Paths
DATA_DIR = Path("../data/HCC1395_inputs")
VCF_PATH = DATA_DIR / "annotated.expression.vcf.gz"
HLA_PATH = DATA_DIR / "optitype_normal_result.tsv"
PROTEOME_PATH = DATA_DIR / "Homo_sapiens.GRCh38.pep.all.gz"
# HLAPOLLO_BINARY kept for reference; Docker is used by default
HLAPOLLO_BINARY = Path("../tools/HLApollo/HLA-Apollo")

print("Imports OK")


Imports OK


In [2]:
# Load HLA types
hla_reader = HLAReader()
hla = hla_reader.read_optitype(str(HLA_PATH))
alleles = list(hla.class_i_alleles)
print(f"HLA-I alleles: {alleles}")

# Load variants (all somatic variant types)
vcf_reader = VCFReader(str(VCF_PATH))
variants = vcf_reader.read_variants()
print(f"Loaded {len(variants)} variants")

# Load proteome
print("Loading proteome (may take a moment)...")
proteome = ProteomeDB(str(PROTEOME_PATH))
print("Proteome loaded.")

# Generate candidates for ALL variants with proteome matches
pg_config = PeptideGenerationConfig(peptide_lengths=(8, 9, 10, 11), n_flank_length=10, c_flank_length=10)
generator = PeptideGenerator(pg_config, proteome)
all_candidates = []
for variant in variants:
    all_candidates.extend(generator.generate(variant))

# Pull expression map for later use
expr_map = {v.gene: v.expression for v in variants if v.expression is not None}
print(f"Generated {len(all_candidates)} peptide candidates")


HLA-I alleles: ['HLA-A*29:02', 'HLA-B*45:01', 'HLA-B*82:02', 'HLA-C*06:02']
Loaded 1209 variants
Loading proteome (may take a moment)...


ProteomeError: Cannot open proteome file '../data/HCC1395_inputs/Homo_sapiens.GRCh38.pep.all.gz': [Errno 2] No such file or directory: '../data/HCC1395_inputs/Homo_sapiens.GRCh38.pep.all.gz'

In [ ]:
# Cache key is keyed by candidate count and alleles.
# A change in either (e.g. more variants, different HLA) triggers a cache miss.
from neoantigen_pipeline.results.cache import PredictionCache

cache = PredictionCache("../results/cache")

# MHCflurry
mf_key = cache.make_key("mhcflurry", len(all_candidates), alleles)
mf_df = cache.get(mf_key)

if mf_df is None:
    print("Running MHCflurry (results will be cached)...")
    mhcflurry_config = MHCIPredictionConfig(alleles=tuple(alleles))
    mhcflurry = MHCflurryPredictor(mhcflurry_config)
    mf_results = mhcflurry.predict_with_processing(all_candidates, alleles)
    mf_df = pd.DataFrame([
        {
            "peptide": r.peptide,
            "gene": r.gene,
            "mutation_str": r.mutation_str,
            "peptide_length": len(r.peptide),
            "mf_best_allele": r.best_allele,
            "mf_affinity_nm": r.affinity_nm,
            "mf_presentation_score": r.presentation_score,
            "mf_percentile": r.presentation_percentile,
        }
        for r in mf_results
    ])
    cache.put(mf_key, mf_df)
    print(f"Cached {len(mf_df)} MHCflurry results")
else:
    print(f"Loaded {len(mf_df)} cached MHCflurry results")

mf_df.head()


In [ ]:
# HLApollo via Docker (default). To skip, set docker_image=None.
ap_key = cache.make_key("hlapollo", len(all_candidates), alleles)
ap_df = cache.get(ap_key)
if ap_df is None:
    print("Running HLApollo via Docker (results will be cached, this takes ~10-15 min)...")
    apollo = HLApolloPredictor()
    ap_results = apollo.predict_with_processing(all_candidates, alleles)
    ap_df = pd.DataFrame([
        {
            "peptide": r.peptide,
            "mutation_str": r.mutation_str,
            "ap_best_allele": r.best_allele,
            "ap_score": r.presentation_score,
            "ap_percentile": r.presentation_percentile,
        }
        for r in ap_results
    ])
    cache.put(ap_key, ap_df)
    print(f"Cached {len(ap_df)} HLApollo results")
else:
    print(f"Loaded {len(ap_df)} cached HLApollo results")
ap_df.head()

In [ ]:
# Merge results on peptide + mutation_str (index-matched, so inner join is safe)
if ap_df is not None:
    combined = mf_df.merge(ap_df, on=["peptide", "mutation_str"], how="inner")
    print(f"Combined: {len(combined)} peptides with predictions from both predictors")
else:
    combined = mf_df.copy()
    print("HLApollo not available; analysis will use MHCflurry results only")

combined.head()

## 2. Score distributions

In [ ]:
if ap_df is not None:
    # Raw scores are on incompatible scales — show each predictor separately
    # (left), then compare percentile ranks on the same axis (right).
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(combined["mf_presentation_score"], bins=50, alpha=0.85, color="steelblue")
    axes[0].set_xlabel("Presentation score (0–1 scale)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("MHCflurry score distribution")

    axes[1].hist(combined["ap_score"], bins=50, alpha=0.85, color="darkorange")
    axes[1].set_xlabel("HLApollo score (log-odds, lower = worse)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("HLApollo score distribution")

    # Percentile ranks are comparable across predictors
    axes[2].hist(combined["mf_percentile"], bins=50, alpha=0.7, label="MHCflurry", color="steelblue")
    axes[2].hist(combined["ap_percentile"], bins=50, alpha=0.7, label="HLApollo", color="darkorange")
    axes[2].set_xlabel("Percentile rank (lower = better presented)")
    axes[2].set_ylabel("Count")
    axes[2].set_title("Percentile rank distributions (same scale)")
    axes[2].legend()

    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(mf_df["mf_presentation_score"], bins=50, color="steelblue", alpha=0.8)
    ax.set_xlabel("MHCflurry presentation score")
    ax.set_ylabel("Count")
    ax.set_title("MHCflurry score distribution (HCC1395)")
    plt.show()


## 3. Rank correlation

In [ ]:
if ap_df is not None:
    rho, pval = stats.spearmanr(combined["mf_percentile"], combined["ap_percentile"])
    print(f"Spearman rank correlation (percentile): ρ = {rho:.3f}, p = {pval:.2e}")

    # With thousands of points a plain scatter is overplotted — use hexbin
    fig, ax = plt.subplots(figsize=(8, 7))
    hb = ax.hexbin(
        combined["mf_percentile"],
        combined["ap_percentile"],
        gridsize=50, cmap="viridis", mincnt=1,
    )
    plt.colorbar(hb, ax=ax, label="Count")
    ax.set_xlabel("MHCflurry percentile rank")
    ax.set_ylabel("HLApollo percentile rank")
    ax.set_title(f"MHCflurry vs HLApollo percentile rank\nSpearman ρ = {rho:.3f}")
    plt.tight_layout()
    plt.show()
else:
    print("HLApollo predictions not available; skipping rank correlation.")


## 4. Top candidate concordance

In [ ]:
if ap_df is not None:
    # Thresholds span a wider range now that we have ~14K candidates instead of 100
    thresholds = [20, 50, 100, 200, 500]
    print(f"{'Threshold':>12} | {'MHCflurry':>10} | {'HLApollo':>10} | {'Overlap':>8} | {'Jaccard':>8}")
    print("-" * 60)

    for k in thresholds:
        n = min(k, len(combined))
        mf_top = set(combined.nsmallest(n, "mf_percentile")["peptide"])
        ap_top = set(combined.nsmallest(n, "ap_percentile")["peptide"])
        overlap = mf_top & ap_top
        union = mf_top | ap_top
        jaccard = len(overlap) / len(union) if union else 0.0
        print(f"Top {k:>8} | {len(mf_top):>10} | {len(ap_top):>10} | {len(overlap):>8} | {jaccard:>8.3f}")

    # Peptides in MHCflurry top 20 but not HLApollo top 20
    mf_top20 = combined.nsmallest(20, "mf_percentile").set_index("peptide")
    ap_top20 = combined.nsmallest(20, "ap_percentile").set_index("peptide")
    mf_only = mf_top20[~mf_top20.index.isin(ap_top20.index)]
    ap_only = ap_top20[~ap_top20.index.isin(mf_top20.index)]

    print(f"\nIn MHCflurry top 20 but not HLApollo top 20 ({len(mf_only)} peptides):")
    display(mf_only[["gene", "mutation_str", "mf_percentile", "ap_percentile"]].reset_index())

    print(f"\nIn HLApollo top 20 but not MHCflurry top 20 ({len(ap_only)} peptides):")
    display(ap_only[["gene", "mutation_str", "mf_percentile", "ap_percentile"]].reset_index())
else:
    print("HLApollo predictions not available; showing MHCflurry top 20:")
    display(mf_df.nsmallest(20, "mf_percentile")[["peptide", "gene", "mutation_str", "mf_percentile"]])


## 5. Allele-specific comparison

### Allele-specific analysis

**Note on B\*82:02:** This allele is rare in training data for both MHCflurry and HLApollo.
Systematic predictor disagreement for B\*82:02 peptides likely reflects higher model
uncertainty rather than genuine biological differences. Results for this allele should
be interpreted with caution.

In [ ]:
if ap_df is not None:
    allele_corr = []
    for allele in combined["mf_best_allele"].unique():
        subset = combined[combined["mf_best_allele"] == allele]
        if len(subset) < 10:
            continue
        rho, _ = stats.spearmanr(subset["mf_percentile"], subset["ap_percentile"])
        allele_corr.append({"allele": allele, "n": len(subset), "spearman_rho": rho})

    if allele_corr:
        corr_df = pd.DataFrame(allele_corr).sort_values("spearman_rho", ascending=False)

        fig, ax = plt.subplots(figsize=(10, max(3, len(corr_df) * 0.6)))
        bars = ax.barh(corr_df["allele"], corr_df["spearman_rho"], color="steelblue", alpha=0.8)
        ax.set_xlabel("Spearman ρ (MHCflurry vs HLApollo percentile rank)")
        ax.set_title("Predictor agreement by allele")
        ax.axvline(0, color="black", linewidth=0.8)
        for bar, row in zip(bars, corr_df.itertuples()):
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"n={row.n}", va="center", fontsize=10)
        plt.tight_layout()
        plt.show()
    else:
        print("No allele has ≥10 peptides for correlation.")
else:
    print("Allele-specific comparison requires HLApollo predictions.")

## 6. Expression-stratified comparison

In [ ]:
# Expression map is built during candidate generation (expr_map dict)
if ap_df is not None:
    combined["expression"] = combined["gene"].map(expr_map)
    expressed = combined.dropna(subset=["expression"])
    print(f"Peptides with expression data: {len(expressed)}/{len(combined)}")

    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter(
        expressed["mf_presentation_score"],
        expressed["ap_score"],
        c=np.log1p(expressed["expression"]),
        cmap="plasma",
        alpha=0.2,  # low alpha needed at full dataset scale
        s=6,
    )
    plt.colorbar(sc, ax=ax, label="log1p(TPM)")
    ax.set_xlabel("MHCflurry presentation score")
    ax.set_ylabel("HLApollo logit score")
    ax.set_title("Predictor scores coloured by gene expression")
    plt.tight_layout()
    plt.show()
else:
    expressed = mf_df.dropna(subset=["expression"])
    print(f"Peptides with expression data: {len(expressed)}/{len(mf_df)}")
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(
        np.log1p(expressed["expression"]),
        expressed["mf_presentation_score"],
        alpha=0.15, s=6, color="steelblue",
    )
    ax.set_xlabel("log1p(TPM)")
    ax.set_ylabel("MHCflurry presentation score")
    ax.set_title("MHCflurry score vs gene expression")
    plt.tight_layout()
    plt.show()


## 7. Case studies: largest disagreements

In [ ]:
if ap_df is not None:
    combined["rank_diff"] = abs(combined["mf_percentile"] - combined["ap_percentile"])

    # Add expression data; filter to expressed genes only (TPM > 1) so case
    # studies are biologically meaningful
    combined["expression"] = combined["gene"].map(expr_map)
    expressed_mask = combined["expression"] > 1.0
    expressed_combined = combined[expressed_mask].copy()
    print(f"Restricting disagreement analysis to expressed genes (TPM > 1): "
          f"{expressed_mask.sum()}/{len(combined)} peptides")

    cols = ["peptide", "gene", "mutation_str", "expression",
            "mf_best_allele", "ap_best_allele",
            "mf_presentation_score", "mf_percentile",
            "ap_score", "ap_percentile", "rank_diff"]
    top_disagreements = expressed_combined.nlargest(5, "rank_diff")[cols]

    print("\nTop 5 peptides where MHCflurry and HLApollo disagree most (expressed genes):")
    display(top_disagreements)
else:
    print("Case studies require HLApollo predictions.")
    print("\nTop 5 MHCflurry candidates:")
    display(mf_df.nsmallest(5, "mf_percentile")[["peptide", "gene", "mutation_str", "mf_best_allele", "mf_percentile"]])


## 8. ESM-2 features (if available)

In [ ]:
try:
    import esm  # noqa: F401
    import h5py  # noqa: F401
    ESM_AVAILABLE = True
except ImportError:
    ESM_AVAILABLE = False

if ESM_AVAILABLE:
    from neoantigen_pipeline.prediction.esm_embeddings import ESMEmbeddingCache

    cache = ESMEmbeddingCache(cache_path="../results/esm_cache.h5")
    print("ESM-2 available. Computing structural scores for top candidates...")

    rank_col = "ap_percentile" if ap_df is not None else "mf_percentile"
    top_peptides = set(combined.nsmallest(50, rank_col)["peptide"])
    top_candidates = [c for c in all_candidates if c.peptide_sequence in top_peptides]

    esm_scores = []
    for c in top_candidates[:20]:  # limit to 20 for speed
        seq = proteome.get_sequence(c.transcript_id)
        if seq:
            features = cache.extract_peptide_features(
                c.transcript_id, seq,
                start=c.aa_pos - 1,
                end=c.aa_pos - 1 + c.peptide_length,
            )
            esm_scores.append({
                "peptide": c.peptide_sequence,
                "gene": c.gene,
                "esm_magnitude": float(np.linalg.norm(features)),
            })

    esm_df = pd.DataFrame(esm_scores)
    display(esm_df)
else:
    print("ESM-2 not installed. Install with: pip install 'neoantigen-pipeline[esm]'")